# Линейная регрессия: количественная оценка влияния факторов

**Проект «ИИ для учёных».** Практический блокнот к странице алгоритма «Линейная регрессия».

## Что делает метод

Линейная регрессия описывает количественную переменную (отклик) как взвешенную сумму предикторов. Метод отвечает на вопрос «как одна измеряемая величина зависит от других» и даёт интерпретируемые коэффициенты: каждый коэффициент — это ожидаемое изменение отклика при изменении предиктора на единицу при прочих равных.

Метод применим, когда у исследователя есть таблица наблюдений с непрерывным откликом и связь приближённо линейна. Помимо прогноза регрессия даёт стандартные ошибки коэффициентов, доверительные интервалы и проверку гипотез о значимости предикторов.

В этом блокноте мы подберём линейную модель методом наименьших квадратов, проверим её предпосылки по остаткам и разберём, как читать результат. Расчётное время прохождения — около пяти минут.

## Интуиция за методом

Представьте, что вы строите график: по горизонтали — количество осадков за сезон, по вертикали — урожайность пшеницы. Каждая точка — один год наблюдений. Линейная регрессия проводит через это облако точек прямую линию так, чтобы суммарное расстояние от точек до линии было наименьшим. Угол наклона этой прямой и есть найденная закономерность: насколько в среднем вырастает урожайность при увеличении осадков на 10 мм.

Когда предикторов несколько (осадки, температура, доза удобрения), метод делает то же самое, но уже в многомерном пространстве — строит «гиперплоскость», проходящую максимально близко ко всем точкам одновременно. Каждый коэффициент при предикторе показывает его вклад при условии, что все остальные факторы зафиксированы.

**Ключевые термины, которые встретятся ниже:**
- **Предиктор (признак)** — измеримая характеристика объекта, которую мы используем для прогноза (например, температура, доза, концентрация).
- **Отклик** — величина, которую мы хотим предсказать (например, урожайность, выход продукта).
- **Обучающая выборка** — часть данных, на которой модель подбирает коэффициенты.
- **Тестовая выборка** — отложенная часть данных, на которой мы проверяем, как модель работает на новых наблюдениях.
- **Остаток** — разница между наблюдённым и предсказанным значением отклика.

## 1. Установка библиотек

В среде Google Colab перечисленные библиотеки, как правило, уже установлены. Ячейка ниже гарантирует их наличие, в том числе при локальном запуске.

In [ ]:
%pip install -q scikit-learn numpy pandas matplotlib statsmodels

## 2. Единый стиль графиков

Все графики в блокнотах проекта оформляются в едином визуальном стиле сайта «ИИ для учёных»: общая палитра, шрифты, толщины линий и сетка. Ниже встроено содержимое стилевого шаблона `notebooks/viz_style.py` — отдельный файл загружать не нужно. Вызов `apply_viz_style()` настраивает matplotlib один раз на весь блокнот.

In [ ]:
# Единый стилевой шаблон графиков проекта «ИИ для учёных».
# Значения синхронизированы с токенами темы сайта (styles/tokens.css).
VIZ_TOKENS = {
    "background": "#ffffff",  # фон полотна
    "node_fill":  "#eef1f6",  # заливка блоков
    "node_text":  "#0e1726",  # основной текст
    "edge":       "#4d5e78",  # линии, оси, подписи
    "grid":       "#dce2ec",  # сетка координат
    "series":     ["#2563eb", "#0d9488", "#b45309", "#4d5e78"],  # цвета рядов
}
VIZ = VIZ_TOKENS


def apply_viz_style():
    """Настраивает matplotlib под единый стиль сайта."""
    import matplotlib as mpl
    from cycler import cycler
    t = VIZ_TOKENS
    mpl.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Segoe UI", "DejaVu Sans", "Arial", "Helvetica"],
        "font.size": 12, "axes.titlesize": 15, "axes.titleweight": "semibold",
        "axes.labelsize": 13, "xtick.labelsize": 11, "ytick.labelsize": 11,
        "legend.fontsize": 11,
        "figure.facecolor": t["background"], "axes.facecolor": t["background"],
        "savefig.facecolor": t["background"], "figure.dpi": 110, "savefig.dpi": 150,
        "axes.prop_cycle": cycler(color=t["series"]),
        "text.color": t["node_text"], "axes.labelcolor": t["node_text"],
        "axes.titlecolor": t["node_text"], "axes.edgecolor": t["edge"],
        "xtick.color": t["edge"], "ytick.color": t["edge"], "axes.linewidth": 1.2,
        "axes.grid": True, "grid.color": t["grid"], "grid.linewidth": 1.0,
        "grid.linestyle": "-", "axes.axisbelow": True,
        "lines.linewidth": 2.0, "lines.markersize": 6, "patch.linewidth": 1.5,
        "axes.spines.top": False, "axes.spines.right": False,
        "legend.frameon": False,
    })


def get_palette(n=None):
    """Возвращает список цветов рядов из токенов темы."""
    series = VIZ_TOKENS["series"]
    if n is None:
        return list(series)
    return [series[i % len(series)] for i in range(n)]


apply_viz_style()
print("Единый стиль графиков подключён.")

## 3. Демонстрационные данные

Используем открытый набор по диабету из `scikit-learn`: 442 пациента, 10 числовых признаков (возраст, индекс массы тела, давление, биохимические показатели) и количественный отклик — мера прогрессирования заболевания через год. Задача — оценить, как клинические показатели связаны с прогрессированием.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_diabetes

data = load_diabetes(as_frame=True)
X = data.data            # таблица предикторов
y = data.target          # непрерывный отклик
feature_names = list(X.columns)

print(f'Наблюдений: {X.shape[0]}, предикторов: {X.shape[1]}')
X.head()

### Наглядный «ага»-эксперимент: что такое линия регрессии и остатки

Прежде чем переходить к многомерной модели, посмотрим на суть метода на простом двумерном примере. Возьмём один предиктор (ИМТ) и отклик — прогрессирование диабета. На графике ниже: синяя линия — это и есть линейная регрессия. Вертикальные отрезки — остатки: насколько каждое наблюдение отклоняется от предсказания модели. Метод подобрал линию именно так, чтобы сумма квадратов длин этих отрезков была минимальной.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.linear_model import LinearRegression

# Берём один предиктор для наглядности
x_demo = X['bmi'].to_numpy()
y_demo = y.to_numpy()

reg1d = LinearRegression().fit(x_demo.reshape(-1, 1), y_demo)
x_line = np.linspace(x_demo.min(), x_demo.max(), 200)
y_line = reg1d.predict(x_line.reshape(-1, 1))
y_hat = reg1d.predict(x_demo.reshape(-1, 1))

fig, ax = plt.subplots(figsize=(9.5, 5.5))

# Остатки — вертикальные отрезки от точки до линии
for xi, yi, yhi in zip(x_demo, y_demo, y_hat):
    ax.plot([xi, xi], [yi, yhi],
            color=VIZ['series'][2], linewidth=0.7, alpha=0.5)

ax.scatter(x_demo, y_demo, color=VIZ['series'][0],
           alpha=0.55, s=22, label='наблюдения', zorder=3)
ax.plot(x_line, y_line, color=VIZ['series'][2], linewidth=2.5,
        label='линия регрессии', zorder=4)

ax.set_xlabel('Индекс массы тела (стандартизованный)')
ax.set_ylabel('Прогрессирование диабета через год')
ax.set_title('Линейная регрессия: линия и остатки')
ax.legend()
fig.tight_layout()
plt.show()

print(f'Наклон линии (коэффициент при ИМТ): {reg1d.coef_[0]:.1f}')
print(f'R² для однофакторной модели: {reg1d.score(x_demo.reshape(-1,1), y_demo):.3f}')

**Как читать этот график.** Каждая точка — один пациент. Синяя линия — прогноз модели: для любого значения ИМТ она выдаёт ожидаемое прогрессирование диабета. Оранжевые вертикальные отрезки — остатки: чем они длиннее, тем хуже модель описывает конкретное наблюдение. Наклон линии показывает: чем выше ИМТ, тем в среднем выраженнее прогрессирование. Значение R² под графиком говорит, какую долю разброса отклика объясняет этот единственный предиктор.

## 4. Применение метода

Разделим данные на обучающую и тестовую части и подберём линейную модель методом наименьших квадратов (МНК).

**Зачем делить данные?** Если оценивать качество модели на тех же данных, на которых она обучалась, мы получим чрезмерно оптимистичную картину — модель «видела» эти наблюдения и просто запомнила их. Тестовая выборка — это данные, которые модель не видела ни разу, и её качество на них честно отражает способность обобщаться на новые случаи. Такое переусложнение модели под обучающие данные называется **переобучением**.

**Что делает МНК?** Метод наименьших квадратов минимизирует сумму квадратов остатков — расхождений между наблюдённым и предсказанным значением отклика.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)
print('Модель обучена на', X_train.shape[0], 'наблюдениях.')

### Качество модели и коэффициенты

Следующая ячейка вычисляет две **метрики** — числовые показатели качества модели:

- **R² (коэффициент детерминации)** — доля дисперсии отклика, объяснённая моделью. Варьируется от 0 до 1: 0 означает, что модель ничем не лучше простого среднего; 1 — идеальное описание данных.
- **Средняя абсолютная ошибка (MAE)** — среднее абсолютное расхождение между прогнозом и реальным значением, в единицах измерения отклика.

Также выводятся **коэффициенты** при каждом предикторе: их знак показывает направление влияния (положительный — предиктор повышает отклик), а модуль — силу.

In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error

y_pred = model.predict(X_test)
print(f'R2 на тесте: {r2_score(y_test, y_pred):.3f}')
print(f'Средняя абсолютная ошибка: {mean_absolute_error(y_test, y_pred):.1f}')

coef = pd.Series(model.coef_, index=feature_names)
coef.sort_values()

### Значимость коэффициентов

Точечная оценка коэффициента сама по себе не говорит о его надёжности: нас интересует, не мог ли такой коэффициент возникнуть случайно. Библиотека `statsmodels` даёт:
- **стандартную ошибку** — оценку неопределённости коэффициента,
- **доверительный интервал** [0.025; 0.975] — диапазон правдоподобных значений коэффициента,
- **p-значение** — вероятность получить такой или более крайний коэффициент, если истинный эффект равен нулю; малое p (< 0.05) означает, что влияние предиктора статистически значимо.

Если `statsmodels` недоступна, доверительные интервалы оцениваются **бутстрэпом** — многократным переобучением на случайных подвыборках данных.

In [ ]:
import statsmodels.api as sm

try:
    Xc = sm.add_constant(X_train)
    ols = sm.OLS(y_train, Xc).fit()
    summary = ols.summary2().tables[1][['Coef.', 'Std.Err.', 'P>|t|',
                                        '[0.025', '0.975]']]
    print(summary.round(3))
except Exception as exc:  # фолбэк без statsmodels
    print('statsmodels недоступна, используем бутстрэп:', exc)
    rng = np.random.default_rng(42)
    boot = []
    Xa = X_train.to_numpy(); ya = y_train.to_numpy()
    for _ in range(500):
        idx = rng.integers(0, len(Xa), len(Xa))
        boot.append(LinearRegression().fit(Xa[idx], ya[idx]).coef_)
    boot = np.array(boot)
    ci = np.percentile(boot, [2.5, 97.5], axis=0)
    print(pd.DataFrame({'Coef.': model.coef_,
                        '[0.025': ci[0], '0.975]': ci[1]},
                       index=feature_names).round(3))

### Диагностика остатков

Перед тем как делать содержательные выводы из коэффициентов, нужно проверить, выполняются ли предпосылки метода. Для этого строятся два диагностических графика:

- **Прогноз против факта** — если модель работает хорошо, точки должны лежать вдоль диагональной пунктирной линии.
- **Остатки против предсказаний** — если всё в порядке, точки должны хаотично рассеиваться вокруг нуля без видимой структуры. Видимый изгиб говорит о нелинейности; расширяющийся «рупор» — о гетероскедастичности (неравномерном разбросе ошибок).

In [ ]:
import matplotlib.pyplot as plt

resid = y_test - y_pred
fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.2))

lim = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
axes[0].scatter(y_test, y_pred, alpha=0.6, color=VIZ['series'][0])
axes[0].plot(lim, lim, color=VIZ['series'][2], linestyle='--',
             label='идеальный прогноз')
axes[0].set_xlabel('Наблюдённое значение отклика')
axes[0].set_ylabel('Предсказанное значение')
axes[0].set_title('Прогноз против факта')
axes[0].legend()

axes[1].scatter(y_pred, resid, alpha=0.6, color=VIZ['series'][1])
axes[1].axhline(0, color=VIZ['series'][2], linestyle='--')
axes[1].set_xlabel('Предсказанное значение')
axes[1].set_ylabel('Остаток (факт минус прогноз)')
axes[1].set_title('Остатки против предсказаний')

fig.tight_layout()
plt.show()

**Как читать эти графики.**

Левый: точки должны лежать вдоль оранжевой пунктирной линии «идеального прогноза». Отклонение вверх означает, что модель недооценивает; вниз — переоценивает. Систематическое смещение в одной зоне значений указывает на нелинейность.

Правый: точки должны беспорядочно рассеиваться вокруг горизонтальной нулевой линии. Если разброс увеличивается слева направо (форма рупора) — это гетероскедастичность; если точки образуют дугу — нелинейная зависимость. Оба нарушения делают стандартные ошибки и p-значения коэффициентов ненадёжными.

## 5. Интерпретация результата

- **R²** — доля объяснённой дисперсии отклика; чем ближе к единице, тем точнее модель. Низкое R² означает, что линейная связь слаба или важные предикторы не учтены.
- **Коэффициенты**: знак указывает направление влияния, модуль — силу. Сравнивать модули корректно, только если предикторы в единой шкале (в наборе `diabetes` признаки уже стандартизованы).
- **Доверительный интервал и p-значение**: если интервал не пересекает ноль (p мало), влияние предиктора статистически значимо. Значимость не равна причинно-следственной связи.
- **График остатков**: видимая структура (изгиб, расширяющийся разброс) указывает на нарушение предпосылок — нелинейность или гетероскедастичность.

Важно помнить: экстраполяция за диапазон наблюдённых значений предикторов ненадёжна.

## 7. Поэкспериментируйте сами

Попробуйте изменить параметры блокнота и посмотрите, что произойдёт:

1. **Один предиктор против нескольких.** В ячейке раздела 4 замените `X` на `X[['bmi']]` (только ИМТ) и переобучите модель. Сравните R² с полной моделью — насколько остальные предикторы добавляют объяснительную силу?

2. **Доля тестовой выборки.** Измените `test_size=0.3` на `0.1` (маленький тест) и на `0.7` (большой тест). Как меняется R² на тесте? При очень маленьком тесте оценка качества становится ненадёжной.

3. **Добавьте квадратичный признак.** Выполните код ниже, чтобы добавить ИМТ² в набор предикторов, и снова обучите модель. Вырастет ли R²? Это простейший способ учесть нелинейность в линейной модели.

In [ ]:
# Эксперимент: добавляем квадрат ИМТ как дополнительный признак
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

X_exp = data.data.copy()
X_exp['bmi_sq'] = X_exp['bmi'] ** 2   # квадратичный признак

X_tr, X_te, y_tr, y_te = train_test_split(
    X_exp, data.target, test_size=0.3, random_state=42)

m_exp = LinearRegression().fit(X_tr, y_tr)
print(f'R² с квадратичным признаком: {r2_score(y_te, m_exp.predict(X_te)):.3f}')
print('(Сравните с R² исходной модели из раздела 4)')

## 6. Попробуйте на своих данных

Замените демонстрационный набор своими данными. Подготовьте таблицу, где строки — наблюдения, столбцы — числовые предикторы, и отдельный столбец — непрерывный отклик.

1. Загрузите файл в Colab: панель слева, вкладка «Файлы», кнопка загрузки.
2. Снимите комментарии в ячейке ниже и укажите имя файла и название столбца с откликом.
3. Последовательно выполните ячейки разделов 4 и 5 — код переиспользуется без изменений.

In [ ]:
# --- Шаблон загрузки своих данных ---
# df = pd.read_csv('my_data.csv')          # ваш файл
# target_column = 'отклик'                  # имя столбца с откликом
#
# y = df[target_column]
# X = df.drop(columns=[target_column]).select_dtypes('number')
# feature_names = list(X.columns)
#
# print(f'Наблюдений: {X.shape[0]}, предикторов: {X.shape[1]}')
# Далее повторите ячейки раздела 4.

## 8. Краткие выводы

- Линейная регрессия строит наилучшую прямую (или гиперплоскость) через облако данных, минимизируя суммарное квадратичное отклонение точек от неё.
- Каждый коэффициент показывает ожидаемое изменение отклика при изменении предиктора на единицу при прочих равных — это делает метод интерпретируемым.
- R² (доля объяснённой дисперсии) и MAE (средняя ошибка) измеряют качество прогноза; оценивать их нужно на тестовой выборке, которую модель не видела.
- Диагностика остатков обязательна: нарушение предпосылок (нелинейность, гетероскедастичность) делает p-значения и доверительные интервалы ненадёжными.
- При большом числе коррелированных предикторов переходите к регуляризованной регрессии (Ridge, Lasso, ElasticNet) — следующий блокнот серии.

## Три вопроса для самопроверки

Ответьте до того, как раскроете подсказку, — это проверяет, что метод понят, а не просто запущен.

1. На графике «Остатки против предсказаний» точки образуют расширяющийся «рупор»: при малых предсказанных значениях разброс остатков невелик, при больших — значительно шире. Что это нарушение означает для интерпретации p-значений и доверительных интервалов коэффициентов?
2. Два предиктора в таблице коэффициентов имеют противоположные знаки и оба статистически значимы, но вы знаете, что в предметной области они должны действовать в одном направлении. Назовите наиболее вероятную причину и укажите, как её диагностировать.
3. Модель обучена на пациентах с ИМТ от 18 до 35. Исследователь хочет использовать её для оценки прогрессирования диабета у пациента с ИМТ = 45. Почему это ненадёжно, и какой диагностический инструмент поможет выявить подобный выход за пределы обучающего диапазона?

<details>
<summary>Показать ориентиры для ответов</summary>

1. Это гетероскедастичность — нарушение предпосылки о постоянстве дисперсии остатков. При её наличии стандартные ошибки МНК занижены или завышены неравномерно, поэтому p-значения и доверительные интервалы ненадёжны. Следует применить взвешенный МНК или робастные стандартные ошибки (HC3).
2. Наиболее вероятная причина — мультиколлинеарность: два сильно скоррелированных предиктора «перетягивают» коэффициенты друг у друга. Диагностика: вычислите VIF (variance inflation factor) для каждого предиктора; значение VIF > 5–10 указывает на проблему.
3. Линейная регрессия экстраполирует линейно в область, где форма зависимости неизвестна, а обучающих данных нет — прогноз ненадёжен. Для обнаружения таких точек используют leverage (мера удалённости от центра обучающих данных): наблюдение с высоким leverage находится далеко от обучающего облака и требует особой осторожности.
</details>
